In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
#from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib import colors
from matplotlib.ticker import LinearLocator, FormatStrFormatter
import pandas as pd
import yt
#from yt.frontends.boxlib.data_structures import AMReXDataset 
#yt.enable_parallelism()
yt.set_log_level(0) # do not show log output

In [ ]:
pathname = '/Users/sonnen/Codes/gempic_home/gempic_runs/Bernstein' # Add the path to your data folder
pathname_out = pathname + '/processed'
try:
    os.mkdir(pathname_out)
except(FileExistsError):
    pass
os.chdir(pathname)
sim_name = "rho"
# read times series
ts = yt.load('./fullDiagnostics/' + sim_name + '??????')
ntz = ts.__len__() # number of items in time series

# print field list and choose field to be used for dispersion relation
print(ts[0].field_list)


In [ ]:
# read in the data for each time slice
field = 'rho'
storage = {}
for store, ds in ts.piter(storage=storage):
    data = ds.covering_grid( 0, ds.domain_left_edge, ds.domain_dimensions )
    arr = np.array(data['boxlib',field])
    store.result = np.sum(np.sum(arr,2),1); # we sum over the y and z components
    time = float(ds.current_time)
    #print(time)

In [ ]:
# get space dimensions
nx, ny, nz = ts[0].domain_dimensions
x_left = np.array(ds.domain_left_edge)
x_right = np.array(ds.domain_right_edge)
L = x_right - x_left
# fill array for FFT
arr = np.zeros([nx,ntz]);
print(arr.shape)
for data in storage.items():
    arr[:,data[0]] = data[1]    

# apply the hann filter
hann = np.hanning(ntz);
arr = arr * hann
# FFT in space and time
arrfft = np.fft.fftn(arr)
print("fft ",arrfft[0,0])
# normalize and transpose FFT array for plots
arrfftnorm = np.transpose(np.abs(arrfft))/np.abs(arrfft).max()
[N,L] = arrfftnorm.shape 

T = time # last current time
Lx = x_right[0] - x_left[0]
print(ntz,N,L,T,2*np.pi/Lx,2*np.pi/T)

# define frequency (om) and wave number (kx) values
Lmax = int(L/2)
Nmax = int(N/2)
om = 2*np.pi/T * np.arange(Nmax)
kx = 2*np.pi/Lx * np.arange(Lmax)
# subrange for plotting
Lmax = int(L/4)
Nmax = int(N/32)
om = 2*np.pi/T * np.arange(Nmax)
kx = 2*np.pi/Lx * np.arange(Lmax)
kmax= int(Lmax/4)
omax = int(Nmax/4)
#print(om)

In [ ]:
from scipy.special import i0, iv  # modified bessel function
from scipy.optimize import brenth
# Bernstein wave dispersion function from Bernstein 1958 (Z is approximated)
# also Mario Raeth's thesis formula (4.1.4)
def D(om):
    s = np.exp(-k**2)*iv(0,k**2) - 2 #k**2 (no k**2 in QN case)
    for n in range(1,50):
        s = s + om * np.exp(-k**2)*iv(n,k**2) *(1/(om - n) + 1/(om+n))
    return s
nom = int(om[-1])
roots = np.zeros((nom,Lmax))
for i in range(1,Lmax):
    k = kx[i-1]
    for j in range (nom):
        try:
            roots[j,i] = brenth(D,j+1.001,j+1.999)
        except(ValueError):
            roots[j,i] = 0

In [ ]:
lvls = np.logspace(-9., 0, 100)
#lvls = np.logspace(-6.0, -2, 20)
plt.cla()

#plt.contourf(kx[0:kmax], om[0:omax], a[0:kmax, 0:omax], cmap=cm.jet,norm=colors.LogNorm(), levels=lvls)
plt.contourf(kx[1:], om, arrfftnorm [0:Nmax, 1:Lmax], cmap=cm.jet,norm=colors.LogNorm(), levels=lvls)
plt.colorbar()
#plt.xlabel('wave number' r'$\ k \Delta x$',fontsize=13)
plt.xlabel('wave number' r'$\ k$',fontsize=13)
plt.ylabel('frequency' r'$ \ \omega$',fontsize=14)
# plot analytical solutions
for i in range(nom):
    plt.plot(kx,roots[i,:],'k.',markersize=2)

plt.savefig(pathname_out+'/dispersion_relation')


In [ ]:
# plot electric energy
tabE=pd.read_csv("reducedDiagnostics/Een.txt",delim_whitespace=True)
#tab.plot(1,2)
t = tabE.values[:,1]
ex2 = tabE.values[:,2]
ey2 = tabE.values[:,3]
ez2 = tabE.values[:,4]
etot = ex2+ey2+ez2
#plt.plot(t,etot)
# plot particle kinetic energy
tabK=pd.read_csv("reducedDiagnostics/Kin.txt",delim_whitespace=True)
#tab.plot(1,2)
t = tabK.values[:,1]
kin = tabK.values[:,2]
plt.plot(t,kin)